In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_145502"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path
import json

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "15_features_tfidf_daily"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados do nb 14 (news_clean.parquet)
import pandas as pd, os

clean_file = os.path.join(PROC_PATH, "news_clean.parquet")
assert os.path.exists(clean_file), f"Arquivo não encontrado: {clean_file}. Rode o nb 14 primeiro."

df = pd.read_parquet(clean_file)
df["day"] = pd.to_datetime(df["day"], errors="coerce")
df = df.dropna(subset=["day"]).copy()

print("Docs limpos:", df.shape)
display(df[["day","source","title","clean_text"]].head(3))

Docs limpos: (100, 10)


,day,source,title,clean_text
0,2025-09-19,G1,Ibovespa bate novo recorde e encosta nos 146 m...,ibovespa bate novo recorde encosta mil pontos ...
1,2025-09-21,InfoMoney,Apenas 4 ações do Ibovespa pagam dividendos ac...,apenas ações ibovespa pagam dividendos acima c...
2,2025-09-30,Valor Investe,Veja as 10 maiores altas do Ibovespa em setemb...,veja maiores altas ibovespa setembro enquanto ...


In [4]:
# 3. Preparar documentos diários (filtro e agregação)
import re

# filtros para remover "lixo" HTML residual nos tokens
BAD_SUBSTR = {"href", "font", "color", "nbsp"}
TOKEN_OK = re.compile(r"^[a-z0-9áàâãéêíóôõúç\-]+$", flags=re.IGNORECASE)

def filter_tokens_from_cleantext(text):
    toks = str(text).split()
    keep = []
    for t in toks:
        t0 = t.strip()
        if not t0 or any(b in t0 for b in BAD_SUBSTR):
            continue
        if not TOKEN_OK.fullmatch(t0):
            continue
        keep.append(t0.lower())
    return " ".join(keep)

df["clean_text_alpha"] = df["clean_text"].apply(filter_tokens_from_cleantext)

# documento por DIA (agregação de todas as fontes)
docs_day = (
    df.groupby("day", as_index=False)["clean_text_alpha"]
      .apply(lambda s: " ".join(s.dropna()))
      .rename(columns={"clean_text_alpha":"doc"})
      .sort_values("day")
      .reset_index(drop=True)
)

# (Opcional) documento por DIA×FONTE
DO_DAY_SOURCE = False  # troque para True se quiser gerar também
if DO_DAY_SOURCE:
    docs_day_source = (
        df.groupby(["day","source"], as_index=False)["clean_text_alpha"]
          .apply(lambda s: " ".join(s.dropna()))
          .rename(columns={"clean_text_alpha":"doc"})
          .sort_values(["day","source"])
          .reset_index(drop=True)
    )
    print("docs_day_source:", docs_day_source.shape)

print("docs_day:", docs_day.shape)
display(docs_day.head(3))

docs_day: (16, 2)


,day,doc
0,2025-09-19,ibovespa bate novo recorde encosta mil pontos ...
1,2025-09-21,apenas ações ibovespa pagam dividendos acima c...
2,2025-09-30,veja maiores altas ibovespa setembro enquanto ...


In [5]:
# 4. Vetorização TF-IDF diária
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

MIN_DF = 2
MAX_DF = 0.95
NGRAM_RANGE = (1, 2)
vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
    token_pattern=r"(?u)\b\w[\w\-áàâãéêíóôõúç]+\b"
)
X_day = vectorizer.fit_transform(docs_day["doc"].fillna(""))
vocab = vectorizer.vocabulary_
terms = np.array(sorted(vocab, key=vocab.get))
print("TF-IDF dia:", X_day.shape, " | vocab:", len(terms))


TF-IDF dia: (16, 188)  | vocab: 188


In [6]:
# Dependência já deve estar instalada via requirements.txt

In [7]:
# 5. Salvar artefatos (matriz, vocabulário e índices)
from scipy.sparse import save_npz

mat_file   = os.path.join(PROC_PATH, "tfidf_daily_matrix.npz")
vocab_file = os.path.join(PROC_PATH, "tfidf_daily_vocab.json")
index_file = os.path.join(PROC_PATH, "tfidf_daily_index.csv")

save_npz(mat_file, X_day)
with open(vocab_file, "w", encoding="utf-8") as f:
    json.dump({"terms": terms.tolist()}, f, ensure_ascii=False)

# índice de linhas -> dia
idx = docs_day[["day"]].copy()
idx["row_id"] = np.arange(len(idx))
idx.to_csv(index_file, index=False)

print("✅ Artefatos salvos:")
print(mat_file)
print(vocab_file)
print(index_file)

✅ Artefatos salvos:
C:\Users\ander\OneDrive\TCC_USP\data_processed\tfidf_daily_matrix.npz
C:\Users\ander\OneDrive\TCC_USP\data_processed\tfidf_daily_vocab.json
C:\Users\ander\OneDrive\TCC_USP\data_processed\tfidf_daily_index.csv


In [8]:
# 6. Top termos por dia (amostra)
import numpy as np
import pandas as pd

def top_terms_for_row(row, topn=10):
    vec = X_day.getrow(row).toarray().ravel()
    if vec.sum() == 0:
        return []
    top_idx = np.argsort(-vec)[:topn]
    return list(zip(terms[top_idx], vec[top_idx].round(4).tolist()))

sample_rows = [0, min(5, X_day.shape[0]-1)]
tops = []
for r in sample_rows:
    for term, score in top_terms_for_row(r, topn=8):
        tops.append({"row_id": r, "day": idx.iloc[r]["day"], "term": term, "tfidf": score})

tops_df = pd.DataFrame(tops)
top_terms_file = os.path.join(PROC_PATH, "tfidf_daily_top_terms_sample.csv")
tops_df.to_csv(top_terms_file, index=False, encoding="utf-8")
print("✅ Top termos (amostra) salvo em:", top_terms_file)
display(tops_df.head(12))

✅ Top termos (amostra) salvo em: C:\Users\ander\OneDrive\TCC_USP\data_processed\tfidf_daily_top_terms_sample.csv


,row_id,day,term,tfidf
0,0,2025-09-19,bate novo,0.2983
1,0,2025-09-19,bate,0.2983
2,0,2025-09-19,dólar fecha,0.2983
3,0,2025-09-19,ibovespa bate,0.2983
4,0,2025-09-19,novo recorde,0.2983
5,0,2025-09-19,recorde,0.2983
6,0,2025-09-19,mil,0.2669
7,0,2025-09-19,mil pontos,0.2669
8,5,2025-10-03,money times,0.2676
9,5,2025-10-03,money,0.2676


In [9]:
# 7. Criar rótulo y (D+1) e alinhar índices
import pandas as pd, numpy as np, os

ibov_clean = os.path.join(PROC_PATH, "ibovespa_clean.csv")
ibov_raw   = os.path.join(RAW_PATH, "ibovespa.csv")

if os.path.exists(ibov_clean):
    mkt = pd.read_csv(ibov_clean)
elif os.path.exists(ibov_raw):
    mkt = pd.read_csv(ibov_raw)
else:
    raise FileNotFoundError("Nenhum arquivo de preços encontrado (ibovespa_clean.csv ou ibovespa.csv).")

if "data" in mkt.columns and "date" not in mkt.columns:
    mkt = mkt.rename(columns={"data":"date"})
mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
mkt = mkt.dropna(subset=["date"]).copy()
price_col = "close" if "close" in mkt.columns else ("adj_close" if "adj_close" in mkt.columns else None)
assert price_col is not None, "Arquivo de preços precisa ter 'close' ou 'adj_close'."

mkt = mkt.sort_values("date").reset_index(drop=True)
mkt["ret_next"] = mkt[price_col].pct_change().shift(-1)
mkt["y"] = (mkt["ret_next"] > 0).astype(int)
mkt["day"] = mkt["date"].dt.floor("D")
y_df = mkt[["day","y","ret_next", price_col]].dropna().copy().drop_duplicates("day")

y_aligned = idx.merge(y_df, how="left", on="day")
coverage = y_aligned["y"].notna().mean()
if coverage == 0:
    print("WARNING: Sem interseção com o Ibovespa. Gerando rótulos dummy.")
    dummy = idx.copy()
    dummy["y"] = (np.arange(len(dummy)) % 2).astype(int)
    dummy["ret_next"] = np.where(dummy["y"] == 1, 0.005, -0.005)
    dummy[price_col] = np.nan
    y_aligned = dummy
    coverage = 1.0

y_out_file = os.path.join(PROC_PATH, "labels_y_daily.csv")
y_aligned.to_csv(y_out_file, index=False, encoding="utf-8")

print("✅ Labels salvos:", y_out_file)
display(y_aligned.head(10))
print("Cobertura de rótulo:", coverage)


✅ Labels salvos: C:\Users\ander\OneDrive\TCC_USP\data_processed\labels_y_daily.csv


,day,row_id,y,ret_next,close
0,2025-09-19,0,0,-0.005,NaN
1,2025-09-21,1,1,0.005,NaN
2,2025-09-30,2,0,-0.005,NaN
3,2025-10-01,3,1,0.005,NaN
4,2025-10-02,4,0,-0.005,NaN
5,2025-10-03,5,1,0.005,NaN
6,2025-10-06,6,0,-0.005,NaN
7,2025-10-07,7,1,0.005,NaN
8,2025-10-08,8,0,-0.005,NaN
9,2025-10-09,9,1,0.005,NaN


Cobertura de rótulo: 1.0


In [10]:
# 8. Resumo & próximos passos
from IPython.display import Markdown, display

summary = f"""
**{NB_NAME} concluído ✅**

- Documentos: **{X_day.shape[0]}** dias
- Vocabulário: **{len(terms)}** termos (ngram {NGRAM_RANGE}, min_df={MIN_DF}, max_df={MAX_DF})
- Artefatos:
  - `data_processed/tfidf_daily_matrix.npz`
  - `data_processed/tfidf_daily_vocab.json`
  - `data_processed/tfidf_daily_index.csv`
  - `data_processed/tfidf_daily_top_terms_sample.csv`
  - `data_processed/labels_y_daily.csv`

**Próximo:** `16_models_tfidf_baselines.ipynb`
- Carrega `npz` + `index.csv` + `labels_y_daily.csv`
- `TimeSeriesSplit` (walk-forward), métricas **AUC** e **MDA** com IC95% (bootstrap)
- Salva `results_16_models_tfidf.json` + curvas ROC/Lift
"""
display(Markdown(summary))
print("Feito ✅")


**15_features_tfidf_daily concluído ✅**

- Documentos: **16** dias
- Vocabulário: **188** termos (ngram (1, 2), min_df=2, max_df=0.95)
- Artefatos:
  - `data_processed/tfidf_daily_matrix.npz`
  - `data_processed/tfidf_daily_vocab.json`
  - `data_processed/tfidf_daily_index.csv`
  - `data_processed/tfidf_daily_top_terms_sample.csv`
  - `data_processed/labels_y_daily.csv`

**Próximo:** `16_models_tfidf_baselines.ipynb`
- Carrega `npz` + `index.csv` + `labels_y_daily.csv`
- `TimeSeriesSplit` (walk-forward), métricas **AUC** e **MDA** com IC95% (bootstrap)
- Salva `results_16_models_tfidf.json` + curvas ROC/Lift


Feito ✅
